# [5회차] ML 과제 - 신용카드 사기 탐지 (Credit Card Fraud Detection)

Kaggle의 Credit Card Fraud Detection 데이터셋을 활용해 클래스 불균형 상황에서의 이진 분류 문제를 다룹니다.
SMOTE 오버샘플링, 모델 학습, 하이퍼파라미터 튜닝, Threshold 조정을 통해 Recall/F1/PR-AUC 목표를 달성하는 것이 목표입니다.

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, average_precision_score,
    precision_score, recall_score, f1_score
)

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42

## 1. 데이터 로드 및 기본 탐색

In [4]:
df = pd.read_csv('creditcard.csv')
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

In [6]:
df.describe()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,284807.000000,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,284807.000000,284807.000000
mean,94813.859575,1.175161e-15,3.384974e-16,-1.379537e-15,2.094852e-15,1.021879e-15,1.494498e-15,-5.620335e-16,1.149614e-16,-2.414189e-15,2.238554e-15,1.724421e-15,-1.245415e-15,8.238900e-16,1.213481e-15,4.866699e-15,1.436219e-15,-3.768179e-16,9.707851e-16,1.036249e-15,6.418678e-16,1.628620e-16,-3.576577e-16,2.618565e-16,4.473914e-15,5.109395e-16,1.686100e-15,-3.661401e-16,-1.227452e-16,88.349619,0.001727
std,47488.145955,1.958696e+00,1.651309e+00,1.516255e+00,1.415869e+00,1.380247e+00,1.332271e+00,1.237094e+00,1.194353e+00,1.098632e+00,1.088850e+00,1.020713e+00,9.992014e-01,9.952742e-01,9.585956e-01,9.153160e-01,8.762529e-01,8.493371e-01,8.381762e-01,8.140405e-01,7.709250e-01,7.345240e-01,7.257016e-01,6.244603e-01,6.056471e-01,5.212781e-01,4.822270e-01,4.036325e-01,3.300833e-01,250.120109,0.041527
min,0.000000,-5.640751e+01,-7.271573e+01,-4.832559e+01,-5.683171e+00,-1.137433e+02,-2.616051e+01,-4.355724e+01,-7.321672e+01,-1.343407e+01,-2.458826e+01,-4.797473e+00,-1.868371e+01,-5.791881e+00,-1.921433e+01,-4.498945e+00,-1.412985e+01,-2.516280e+01,-9.498746e+00,-7.213527e+00,-5.449772e+01,-3.483038e+01,-1.093314e+01,-4.480774e+01,-2.836627e+00,-1.029540e+01,-2.604551e+00,-2.256568e+01,-1.543008e+01,0.000000,0.000000
25%,54201.500000,-9.203734e-01,-5.985499e-01,-8.903648e-01,-8.486401e-01,-6.915971e-01,-7.682956e-01,-5.540759e-01,-2.086297e-01,-6.430976e-01,-5.354257e-01,-7.624942e-01,-4.055715e-01,-6.485393e-01,-4.255740e-01,-5.828843e-01,-4.680368e-01,-4.837483e-01,-4.988498e-01,-4.562989e-01,-2.117214e-01,-2.283949e-01,-5.423504e-01,-1.618463e-01,-3.545861e-01,-3.171451e-01,-3.269839e-01,-7.083953e-02,-5.295979e-02,5.600000,0.000000
50%,84692.000000,1.810880e-02,6.548556e-02,1.798463e-01,-1.984653e-02,-5.433583e-02,-2.741871e-01,4.010308e-02,2.235804e-02,-5.142873e-02,-9.291738e-02,-3.275735e-02,1.400326e-01,-1.356806e-02,5.060132e-02,4.807155e-02,6.641332e-02,-6.567575e-02,-3.636312e-03,3.734823e-03,-6.248109e-02,-2.945017e-02,6.781943e-03,-1.119293e-02,4.097606e-02,1.659350e-02,-5.213911e-02,1.342146e-03,1.124383e-02,22.000000,0.000000
75%,139320.500000,1.315642e+00,8.037239e-01,1.027196e+00,7.433413e-01,6.119264e-01,3.985649e-01,5.704361e-01,3.273459e-01,5.971390e-01,4.539234e-01,7.395934e-01,6.182380e-01,6.625050e-01,4.931498e-01,6.488208e-01,5.232963e-01,3.996750e-01,5.008067e-01,4.589494e-01,1.330408e-01,1.863772e-01,5.285536e-01,1.476421e-01,4.395266e-01,3.507156e-01,2.409522e-01,9.104512e-02,7.827995e-02,77.165000,0.000000
max,172792.000000,2.454930e+00,2.205773e+01,9.382558e+00,1.687534e+01,3.480167e+01,7.330163e+01,1.205895e+02,2.000721e+01,1.559499e+01,2.374514e+01,1.201891e+01,7.848392e+00,7.126883e+00,1.052677e+01,8.877742e+00,1.731511e+01,9.253526e+00,5.041069e+00,5.591971e+00,3.942090e+01,2.720284e+01,1.050309e+01,2.252841e+01,4.584549e+00,7.519589e+00,3.517346e+00,3.161220e+01,3.384781e+01,25691.160000,1.000000


In [7]:
print("Class 비율 (전체 데이터):")
print(df['Class'].value_counts())
print(df['Class'].value_counts(normalize=True))

n_normal = (df['Class'] == 0).sum()
n_fraud = (df['Class'] == 1).sum()
print(f"\n정상 거래(Class=0): {n_normal}건")
print(f"사기 거래(Class=1): {n_fraud}건")

Class 비율 (전체 데이터):
Class
0    284315
1       492
Name: count, dtype: int64
Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64

정상 거래(Class=0): 284315건
사기 거래(Class=1): 492건


전체 284,807건 중 사기 거래는 492건(0.17%)에 불과한 극단적인 클래스 불균형 데이터셋입니다.

## 2. 샘플링

사기 거래(Class=1)는 전부 유지하고, 정상 거래(Class=0)는 10,000건만 `random_state=42`로 무작위 샘플링합니다.

In [8]:
fraud_df = df[df['Class'] == 1]
normal_df = df[df['Class'] == 0].sample(n=10000, random_state=RANDOM_STATE)

sample_df = pd.concat([fraud_df, normal_df], axis=0).reset_index(drop=True)
print("샘플링 후 shape:", sample_df.shape)

샘플링 후 shape: (10492, 31)


In [9]:
print("샘플링 후 Class 비율:")
print(sample_df['Class'].value_counts())
print(sample_df['Class'].value_counts(normalize=True))

샘플링 후 Class 비율:
Class
0    10000
1      492
Name: count, dtype: int64
Class
0    0.953107
1    0.046893
Name: proportion, dtype: float64


샘플링 후 사기 비율이 0.17% → 약 4.7%로 크게 완화되었습니다 (여전히 불균형은 존재).

## 3. 데이터 전처리

`Amount` 변수만 `StandardScaler`로 표준화하여 `Amount_Scaled`로 대체합니다 (V1~V28은 이미 PCA로 스케일링되어 있으므로 별도 처리 불필요).

In [10]:
scaler = StandardScaler()
sample_df['Amount_Scaled'] = scaler.fit_transform(sample_df[['Amount']])
sample_df = sample_df.drop(columns=['Amount'])

X = sample_df.drop(columns=['Class'])
y = sample_df['Class']

print("X shape:", X.shape)
X.columns.tolist()

X shape: (10492, 30)


['Time',
 'V1',
 'V2',
 'V3',
 'V4',
 'V5',
 'V6',
 'V7',
 'V8',
 'V9',
 'V10',
 'V11',
 'V12',
 'V13',
 'V14',
 'V15',
 'V16',
 'V17',
 'V18',
 'V19',
 'V20',
 'V21',
 'V22',
 'V23',
 'V24',
 'V25',
 'V26',
 'V27',
 'V28',
 'Amount_Scaled']

## 4. 학습 데이터와 테스트 데이터 분할

`train_test_split`으로 8:2 분할, `stratify=y`로 클래스 비율을 유지합니다.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train Class 비율:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))
print("\nTest Class 비율:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))

Train Class 비율:
Class
0    7999
1     394
Name: count, dtype: int64
Class
0    0.953056
1    0.046944
Name: proportion, dtype: float64

Test Class 비율:
Class
0    2001
1      98
Name: count, dtype: int64
Class
0    0.953311
1    0.046689
Name: proportion, dtype: float64


## 5. SMOTE 적용

**왜 SMOTE를 적용해야 하는가?**

샘플링 후에도 학습 데이터의 사기 거래 비율은 약 4.7%로 여전히 불균형합니다. 이 상태로 모델을 학습하면
모델이 다수 클래스(정상 거래)를 예측하는 것만으로도 높은 accuracy를 쉽게 달성할 수 있기 때문에,
소수 클래스(사기 거래)의 패턴을 충분히 학습하지 못하고 사기 거래를 정상으로 오분류(Recall 저하)하는
경향이 생깁니다. 특히 사기 탐지 문제는 실제 사기를 놓치는 False Negative의 비용이 매우 크기 때문에
Recall이 중요한데, 단순 accuracy 최적화는 이 목적과 어긋납니다.

SMOTE(Synthetic Minority Over-sampling Technique)는 소수 클래스 샘플들 사이를 보간(interpolation)하여
현실적인 합성 샘플을 생성함으로써 소수 클래스 데이터를 인위적으로 늘립니다. 단순 복제(Random Oversampling)와
달리 새로운 샘플을 생성하기 때문에 모델이 소수 클래스의 결정 경계를 더 잘 학습하도록 돕고, 과적합 위험도
단순 복제보다 낮습니다. 단, SMOTE는 반드시 **학습 데이터에만** 적용하고 테스트 데이터는 원본 분포를
그대로 유지해야 실제 성능을 왜곡 없이 평가할 수 있습니다.

In [12]:
print("SMOTE 적용 전 학습 데이터 Class 비율:")
print(y_train.value_counts())

sm = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

print("\nSMOTE 적용 후 학습 데이터 Class 비율:")
print(y_train_sm.value_counts())

SMOTE 적용 전 학습 데이터 Class 비율:
Class
0    7999
1     394
Name: count, dtype: int64

SMOTE 적용 후 학습 데이터 Class 비율:
Class
0    7999
1    7999
Name: count, dtype: int64


## 6. 모델 학습

RandomForest와 LogisticRegression을 비교해 모델을 선정합니다.
RandomForest는 비선형 패턴 포착에 강하고 SMOTE로 생성된 합성 데이터에 robust하게 동작하는 편이라
사기 탐지 문제에서 자주 쓰이는 baseline 모델입니다.

In [13]:
# LogisticRegression 비교
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_sm, y_train_sm)
lr_proba = lr.predict_proba(X_test)[:, 1]
lr_pred = lr.predict(X_test)

print("=== LogisticRegression ===")
print(classification_report(y_test, lr_pred, digits=4))
print("PR-AUC:", average_precision_score(y_test, lr_proba))

=== LogisticRegression ===
              precision    recall  f1-score   support

           0     0.9970    0.9880    0.9925      2001
           1     0.7931    0.9388    0.8598        98

    accuracy                         0.9857      2099
   macro avg     0.8950    0.9634    0.9261      2099
weighted avg     0.9875    0.9857    0.9863      2099

PR-AUC: 0.9527245239683747


c:\Users\tjrex\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [14]:
# RandomForest (선정 모델)
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)

y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print("=== RandomForest 예측값 (앞 20개) ===")
print(y_pred[:20])
print("\n=== RandomForest 예측 확률 (앞 5개, [정상확률, 사기확률]) ===")
print(rf.predict_proba(X_test)[:5])

=== RandomForest 예측값 (앞 20개) ===
[0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0]

=== RandomForest 예측 확률 (앞 5개, [정상확률, 사기확률]) ===
[[0.98 0.02]
 [1.   0.  ]
 [0.97 0.03]
 [1.   0.  ]
 [0.82 0.18]]


In [15]:
print("=== classification_report (threshold=0.5) ===")
print(classification_report(y_test, y_pred, digits=4))

pr_auc = average_precision_score(y_test, y_proba)
print(f"PR-AUC (average_precision_score): {pr_auc:.4f}")

=== classification_report (threshold=0.5) ===
              precision    recall  f1-score   support

           0     0.9945    0.9975    0.9960      2001
           1     0.9457    0.8878    0.9158        98

    accuracy                         0.9924      2099
   macro avg     0.9701    0.9426    0.9559      2099
weighted avg     0.9922    0.9924    0.9923      2099

PR-AUC (average_precision_score): 0.9537


LogisticRegression은 Class1 F1-score가 목표(0.88)에 못 미치는 반면, RandomForest는 threshold=0.5
기준으로도 Recall=0.8878, F1=0.9158, PR-AUC=0.9537로 이미 세 목표를 모두 상회합니다.
따라서 최종 모델로 **RandomForest**를 선정합니다.

## 7. 최종 성능 평가 (하이퍼파라미터 튜닝 & Threshold 조정)

몇 가지 하이퍼파라미터 조합을 비교하고, Threshold를 조정하며 두 클래스 모두에서
Recall ≥ 0.80, F1 ≥ 0.88, PR-AUC ≥ 0.90을 만족하는지 확인합니다.

In [16]:
candidates = {
    'rf_default (n=100)': rf,  # 이미 학습됨
    'rf_n200_depth10': RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

results = {'rf_default (n=100)': (rf, y_proba, pr_auc)}
for name, clf in candidates.items():
    if name == 'rf_default (n=100)':
        continue
    clf.fit(X_train_sm, y_train_sm)
    proba = clf.predict_proba(X_test)[:, 1]
    ap = average_precision_score(y_test, proba)
    results[name] = (clf, proba, ap)
    print(f"{name}: PR-AUC={ap:.4f}")

print(f"rf_default (n=100): PR-AUC={pr_auc:.4f}")

best_name = max(results, key=lambda k: results[k][2])
best_model, best_proba, best_ap = results[best_name]
print(f"\n최종 선정 모델: {best_name} (PR-AUC={best_ap:.4f})")

rf_n200_depth10: PR-AUC=0.9517
rf_default (n=100): PR-AUC=0.9537

최종 선정 모델: rf_default (n=100) (PR-AUC=0.9537)


Threshold를 0.10 ~ 0.90 범위에서 조정하며 두 클래스 모두 목표(Recall≥0.80, F1≥0.88)를 만족하는 지점을 탐색합니다.

In [17]:
print(f"{'th':>5} | {'c0_prec':>8} {'c0_rec':>8} {'c0_f1':>8} | {'c1_prec':>8} {'c1_rec':>8} {'c1_f1':>8} | meets")
rows = []
for th in np.arange(0.10, 0.91, 0.05):
    y_pred_th = (best_proba >= th).astype(int)
    p0 = precision_score(y_test, y_pred_th, pos_label=0, zero_division=0)
    r0 = recall_score(y_test, y_pred_th, pos_label=0)
    f0 = f1_score(y_test, y_pred_th, pos_label=0)
    p1 = precision_score(y_test, y_pred_th, pos_label=1, zero_division=0)
    r1 = recall_score(y_test, y_pred_th, pos_label=1)
    f1v = f1_score(y_test, y_pred_th, pos_label=1)
    meets = (r0 >= 0.80 and f0 >= 0.88) and (r1 >= 0.80 and f1v >= 0.88)
    rows.append((th, p0, r0, f0, p1, r1, f1v, meets))
    print(f"{th:5.2f} | {p0:8.4f} {r0:8.4f} {f0:8.4f} | {p1:8.4f} {r1:8.4f} {f1v:8.4f} | {meets}")

   th |  c0_prec   c0_rec    c0_f1 |  c1_prec   c1_rec    c1_f1 | meets
 0.10 |   0.9984   0.9130   0.9538 |   0.3532   0.9694   0.5177 | False
 0.15 |   0.9984   0.9475   0.9723 |   0.4750   0.9694   0.6376 | False
 0.20 |   0.9984   0.9655   0.9817 |   0.5793   0.9694   0.7252 | False
 0.25 |   0.9980   0.9795   0.9887 |   0.6963   0.9592   0.8069 | False
 0.30 |   0.9980   0.9840   0.9909 |   0.7460   0.9592   0.8393 | False
 0.35 |   0.9965   0.9875   0.9920 |   0.7845   0.9286   0.8505 | False
 0.40 |   0.9955   0.9915   0.9935 |   0.8396   0.9082   0.8725 | False
 0.45 |   0.9950   0.9945   0.9948 |   0.8889   0.8980   0.8934 | True
 0.50 |   0.9945   0.9975   0.9960 |   0.9457   0.8878   0.9158 | True
 0.55 |   0.9940   0.9980   0.9960 |   0.9556   0.8776   0.9149 | True
 0.60 |   0.9935   0.9985   0.9960 |   0.9659   0.8673   0.9140 | True
 0.65 |   0.9930   0.9995   0.9963 |   0.9882   0.8571   0.9180 | True
 0.70 |   0.9926   0.9995   0.9960 |   0.9881   0.8469   0.9121 | Tru

In [18]:
qualifying = [r for r in rows if r[7]]
print(f"목표를 만족하는 threshold 개수: {len(qualifying)} / {len(rows)}")

# 두 클래스 F1의 합이 최대인 지점을 최종 threshold로 선정 (정상/사기 판별 균형)
final_th, p0, r0, f0, p1, r1, f1v, _ = max(qualifying, key=lambda r: r[3] + r[6])
print(f"\n최종 선정 threshold: {final_th:.2f}")

y_pred_final = (best_proba >= final_th).astype(int)
print("\n=== 최종 classification_report ===")
print(classification_report(y_test, y_pred_final, digits=4))
print(f"PR-AUC: {best_ap:.4f}")

목표를 만족하는 threshold 개수: 9 / 17

최종 선정 threshold: 0.65

=== 최종 classification_report ===
              precision    recall  f1-score   support

           0     0.9930    0.9995    0.9963      2001
           1     0.9882    0.8571    0.9180        98

    accuracy                         0.9929      2099
   macro avg     0.9906    0.9283    0.9571      2099
weighted avg     0.9928    0.9929    0.9926      2099

PR-AUC: 0.9537


In [19]:
final_recall_0, final_f1_0 = r0, f0
final_recall_1, final_f1_1 = r1, f1v
final_pr_auc = best_ap

print("=== 목표 달성 여부 ===")
print(f"Class 0 : Recall={final_recall_0:.4f} (목표>=0.80) {'달성' if final_recall_0>=0.80 else '미달성'} | "
      f"F1={final_f1_0:.4f} (목표>=0.88) {'달성' if final_f1_0>=0.88 else '미달성'}")
print(f"Class 1 : Recall={final_recall_1:.4f} (목표>=0.80) {'달성' if final_recall_1>=0.80 else '미달성'} | "
      f"F1={final_f1_1:.4f} (목표>=0.88) {'달성' if final_f1_1>=0.88 else '미달성'}")
print(f"PR-AUC  : {final_pr_auc:.4f} (목표>=0.90) {'달성' if final_pr_auc>=0.90 else '미달성'}")

=== 목표 달성 여부 ===
Class 0 : Recall=0.9995 (목표>=0.80) 달성 | F1=0.9963 (목표>=0.88) 달성
Class 1 : Recall=0.8571 (목표>=0.80) 달성 | F1=0.9180 (목표>=0.88) 달성
PR-AUC  : 0.9537 (목표>=0.90) 달성


### 결론

최종 모델(RandomForest, n_estimators=100, threshold=0.65 부근)은 다음 결과를 보였습니다.

- **Class 0 (정상)**: Recall ≈ 0.9995, F1 ≈ 0.9963 → 목표(Recall≥0.80, F1≥0.88) **달성**
- **Class 1 (사기)**: Recall ≈ 0.8571, F1 ≈ 0.9180 → 목표(Recall≥0.80, F1≥0.88) **달성**
- **PR-AUC** ≈ 0.9537 → 목표(≥0.90) **달성**

세 지표 모두 Class 0, 1 양쪽에서 목표치를 상회하여 **모든 목표를 달성**했습니다.

**참고 (추가 개선 방향)**: 만약 목표를 달성하지 못했거나 더 개선하고 싶다면 다음을 시도할 수 있습니다.
1. XGBoost/LightGBM 등 부스팅 계열 모델 및 `scale_pos_weight`/`class_weight` 옵션 활용
2. GridSearchCV/Optuna를 통한 보다 폭넓은 하이퍼파라미터 탐색 (max_depth, min_samples_leaf, n_estimators 등)
3. SMOTE 외 ADASYN, BorderlineSMOTE 등 다른 오버샘플링 기법 비교
4. Feature engineering (Time 변수를 시간대별 파생 변수로 변환 등)
5. Precision-Recall Curve 기반으로 비즈니스 비용(오탐 vs 미탐 비용)을 고려한 threshold 최적화